In [ ]:
from torch.nn import *
from IPython.display import clear_output
import torch
import numpy as np
import matplotlib.pyplot as plt
class Res(Sequential):
    def forward(s, _):
        return super().forward(_) + _
class Add(Sequential):
    def forward(s, _):
        return sum(s(_)for s in s)
class T(Sequential):
    def forward(s, _):
        return super().forward(_.mT).mT
class RoPE(Linear):
    c = {}
    def __init__(s, i):
        super().__init__(i, i, 0)
    def forward(s, _):
        c = s.c.get(_.size(-2))
        if c is None:
            c = torch.arange(_.size(-2),dtype=_.dtype,device=_.device)[:,None]*torch.logspace(0,-4,_.size(-1)//2,dtype=_.dtype,device=_.device)
            c = torch.cat((c.sin(), c.cos_()), 1)
            s.c[_.size(-2)] = c
        return c @ s.weight.T + _
class Attn(Linear):
    def __init__(s, i, h, H, g, r):
        super().__init__(i, h*g if h<i else i*g)
        s.k = Linear(i, h) if h<i else lambda _:_
        s.v = Linear(i, H) if H<i else lambda _:_
        s.r = r
    def forward(s, q):
        v = functional.pad(q,(0,0,s.r//2,(s.r+1)//2))
        k = s.k(v)
        v = s.v(v)
        k = k.as_strided((*q.shape[:-1],k.size(-1),s.r),(*k.stride(),k.stride(-2)))
        v = v.as_strided((*q.shape[:-1],v.size(-1),s.r),(*v.stride(),v.stride(-2))).mT
        return ((super().forward(q).unflatten(-1,(-1,k.size(-2))) @ k).softmax(-1) @ v).flatten(-2)
class SwiGLU(Linear):
    def __init__(s, i, h):
        super().__init__(i, h)
        s._ = Linear(i, h)
    def forward(s, _):
        return functional.silu(super().forward(_)) * s._(_)
def M(i, *hH0H1GarR):
    m = []
    for hH0H1GarR in hH0H1GarR:
        if isinstance(hH0H1GarR,tuple):
            h,H0,H1,G,a,r,R = hH0H1GarR
            m += Res(
                Add(
                    Sequential(
                        Linear(i, h, 0),
                        T(Conv1d(h, h, r, padding=r//2, groups=h, bias=0)),
                        Linear(h, H1*G, 0) if R else Identity(),
                    ) if r else Identity(),
                    *(Attn(i, H0, H1, G, R),) if R else (),
                ),
                LayerNorm(H1*G if R else h, elementwise_affine=0),
                SwiGLU(H1*G if R else h, a),
                Linear(a, i, 0),
            ),
        else:
            s,h = hH0H1GarR[:2]
            m += Res(
                Linear(i, h, 0),
                T(MaxPool1d(s)),
                M(*hH0H1GarR[1:]),
                T(ConvTranspose1d(h, h, s, s, groups=h, bias=0)),
                Linear(h, i, 0),
            ),
    return Sequential(*m)
a = torch.from_numpy(np.memmap('py.bin', mode='r'))
a = a.as_strided_((len(a)//1024-1,2048),(1024,1))
m = Sequential(
    Embedding(92, 48),
    RoPE(48),
    torch.compile(M(48,
        (96,0,0,0,192,7,0),
        [2,48,
            (96,48,48,2,192,7,8),
            [2,48,
                (96,48,48,2,192,7,8),
            ],
            (96,48,48,2,192,7,8),
        ],
        (96,0,0,0,192,7,0),
    )),
    Linear(48, 92),
).to('cuda',torch.bfloat16)
o = torch.optim.Adam(m.parameters(), 1/1024)

In [ ]:
L,A = [],[]
for y in range(500):
    for y in range(1):
        lc = 0
        for y in range(20):
            for y in range(20):
                t = a[torch.randint(len(a),[16])].to('cuda',torch.long)
                x = t.put(torch.randint(t.numel(),[t.numel()*3//4],device='cuda'),torch.randint(92,[t.numel()*3//4],device='cuda'))
                y = m(x)
                l = functional.cross_entropy(y.mT, t)
                (l*65536).backward()
                o.step()
                o.zero_grad()
            lc += l.data
        L.append(lc[None])
        A.append((y.argmax(-1)==t).sum()[None]/t.numel())
    plt.subplot(121).plot((L:=[torch.cat(L)])[0].to('cpu',torch.float), label=f'{L[0][-1]:.2f}')
    plt.legend()
    plt.subplot(122).plot((A:=[torch.cat(A)])[0].to('cpu',torch.float), label=f'{A[0][-1]:.2f}')
    plt.legend()
    clear_output(True)
    plt.show()